# Nobel Microorganisms Influencing Corrosion

# 1. Introduction

__Inverse Patterns - Protective Bacteria?__
During the statistical analysis notebook 3, important patterns emerged with some bacteria especially persistent on the normal operation systems this is the less affected by corrosion systems, therefore it was teoritized that some of those could be producing metabolites that inhibith corrosion. Hereby this inverse patterns are studied. Studies had shown that some compounds reported to produce inhibition of corrosion. Those compounds could be sintetised by some bacteria, and for that to occur some enzymes are required, only the bacteria with those enzymes are able to sintetise such compounds. The second part of this notebook would engage with investigating the enzymes present on those inverse_results to see if some of them are possible on the duty of inhibit corrosion or are simple bacteria that happend to be on the environment without being involve on any corrosion bussiness. Notably bacteria phenil  is present,  Phenylalanine and related compounds (like some phenolic acids) have been studied for their corrosion inhibition properties. The bacteria producing these compounds could be offering a natural corrosion protection mechanism. This is similar to how certain biofilms can sometimes act as a protective barrier rather than accelerating corrosion.
For your protective factor analysis, I would recommend:

Extract proteins from the inverse pattern bacteria that are associated with:

Amino acid biosynthesis (particularly aromatic amino acids like phenylalanine, tyrosine, tryptophan)
EPS (extracellular polymeric substance) production
Production of siderophores (which can complex with iron and potentially reduce corrosion)
Organic acid metabolism (some can form protective films)
Proteins involved in oxygen consumption (creating less oxidizing conditions)


Compare protein profiles between sites with different corrosion rates - sites with lower corrosion might have higher abundance of these protective proteins
Look for proteins involved in antagonistic relationships with known corrosion-promoting bacteria

The idea of finding proteins that synthesize protective compounds is excellent. You could specifically search for proteins in pathways like:

Shikimate pathway (leads to aromatic amino acids)
Phenylpropanoid metabolism
Polyamine synthesis
Certain secondary metabolite pathways

__Novel Candidates Microorganisms Inducint Corrosion__
During the statistical feature analysis in Notebook 3, several bacterial genera were identified that showed strong correlations with the high-risk corrosion failure category. This led to hypothesize the potential of  of this microorganisms to be associated with corrosion processes and which were not previously documented.
In Notebook 4, we conducted a comprehensive literature review using academic databases to investigate the bacterial genera. The findings confirmed that these microorganisms have no prior documentation in scientific literature linking them to corrosion-related processes.
Now, we aim to deploy the analyze_protein_hierarchy function to examine whether these bacteria express proteins that are known to induce or accelerate corrosion. This analysis will help determine if these genera possess the molecular machinery to contribute to corrosion despite their absence from corrosion-related literature, potentially identifying novel microbial contributors to corrosion processes that have been overlooked in previous research.


# 2. Importing Libraries and Data Preparation

In [ ]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.cm as cm
from matplotlib.lines import Line2D
from matplotlib.patches import Patch
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import networkx as nx
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import DBSCAN
from scipy.cluster.hierarchy import linkage, dendrogram
import umap
import dash
from dash import dcc, html, Input, Output, State
import community as community_louvain

In [ ]:
sys.path.append(os.path.abspath('..'))  # Ensures the project root is in Python's search path

if Path("/kaggle").exists():
    
    # Create directory structure
    !mkdir -p corrosion_scoring
    
    # Download the necessary files each session always
    !wget -O corrosion_scoring/__init__.py https://raw.githubusercontent.com/MagicAlex238/2_Micro/main/corrosion_scoring_root/corrosion_scoring/__init__.py
    !wget -O corrosion_scoring/global_terms.py https://raw.githubusercontent.com/MagicAlex238/2_Micro/main/corrosion_scoring_root/corrosion_scoring/global_terms.py
    !wget -O corrosion_scoring/scoring_system.py https://raw.githubusercontent.com/MagicAlex238/2_Micro/main/corrosion_scoring_root/corrosion_scoring/scoring_system.py
    !wget -O corrosion_scoring/term_processor.py https://raw.githubusercontent.com/MagicAlex238/2_Micro/main/corrosion_scoring_root/corrosion_scoring/term_processor.py
    # Add current directory to path
    import sys
    sys.path.append(os.getcwd())
    
    # Import package
    import corrosion_scoring as cs
else:
    print("Running in local (VSCode) environment")
    
    ## When in vscode local env first time only
    !pip install git+https://github.com/MagicAlex238/2_Micro.git#subdirectory=corrosion_scoring_root
    import corrosion_scoring as cs

In [ ]:
# Determine the environment
if "google.colab" in sys.modules:
    print("Running in Google Colab environment")
    # for colab
    base_dir = Path("/content/drive/MyDrive/MIC")
    abundance_excel = base_dir / "data_picrust/merged_to_sequence.xlsx"
    output_large = base_dir / "output_large"
    output_base = base_dir
    market_dir = base_dir / "output_large" 
    #Directory to keep some Results
    large_dir = base_dir / "2_Micro/data_visual"
    large_dir.mkdir(parents=True, exist_ok=True)

elif Path("/kaggle").exists():
    print("Running in Kaggle environment")
    # For Kaggle work# Input datasets (read-only in Kaggle) 
    base_dir = Path("/kaggle/input/")  
    abundance_excel = base_dir / "new-picrust/merged_to_sequence.xlsx" 
    #Input physicochemical variables
    data_physicochemical = base_dir / "physicochemical-parameters/Physicochemical.xlsx"
    combined_input = base_dir  / "combined_markers.xlsx"
    #===============================================
    #Directory to keep  Results
    output_base = Path("/kaggle/working/")
    shared_dir = output_base/"Visualisations"
    shared_dir.mkdir(parents=True, exist_ok=True)
    combined_path = output_base / "combined_markers.xlsx"
      
else:
    print("Running in local (VSCode) environment")
    base_dir = Path("data")
    base_dir.mkdir(parents=True, exist_ok=True)
    # Base Paths for local environment
    abundance_excel = base_dir / "merged_to_sequence.xlsx"
    #Input physicochemical variables8ikk                           ´
    data_physicochemical = Path("/home/beatriz/MIC/1_Physicochemical/Data/Physicochemical.xlsx")
    #================================================
    # This files are too large for github and are store on Kaggle for educational purposes
    output_large = Path("/home/beatriz/MIC/output_large")
    output_base = base_dir 
    #Directory to keep some Results
    shared_dir= Path("/home/beatriz/SharedFolder/Visualisations/")
    combined_path = base_dir / "combined_markers.xlsx"
       

### Importing the files

In [ ]:
# physicochemical data data from notebook 8 repo 1_Physicochemical
all_physichem = pd.read_excel(data_physicochemical, sheet_name='all_physicochemical', engine ='openpyxl')
# Microbiological Data from notebook 6 repo 2_Micro.
inverse_df = pd.read_excel(combined_path, sheet_name='df_proteins',  engine ='openpyxl')
#mapping of genera to site from the original df for validation
genus_site_df = pd.read_excel(combined_path, sheet_name='genus_to_sites', engine ='openpyxl')
# All markers after pattern, integration, classification, increasing, and balance functions
balanced_markers= pd.read_excel(combined_path, sheet_name='balanced_markers', engine ='openpyxl')

# 3. Inverse Patterns - Protective Bacteria?

inverse patterns are collected on inverse_df, so the idea is to rejoin this part of the data at the end of the analysis to reconstruct the whole data and in doing so have an equal representation of the classes. Otherwise the data would have only class 2 and 3 since the pipeline main aim is to filter our the data. Never the less down the pipeline other 3 sites are lost to balance representation function and because of those belong to the increasing results, it is better to rejoin on the missing sites from classified_markers. That is in order to have a comprehensive class representation for the final Neuro Network. On the other hand the Inverse Pattern contain on inverse_results would also be brough forward to analyse wether some of the enzymes are related to known inhibitors to corrosion failure.


In [ ]:
inverse_df

# 4. Novel Candidates Microorganisms Inducint Corrosion 
During the statistical feature analysis in Notebook 3, several bacterial genera were identified that showed strong correlations with the high-risk corrosion failure category. This led to hypothesize the potential of  of this microorganisms to be associated with corrosion processes and which were not previously documented.
In Notebook 4, we conducted a comprehensive literature review using academic databases to investigate the bacterial genera. The findings confirmed that these microorganisms have no prior documentation in scientific literature linking them to corrosion-related processes.
Now, we aim to deploy the analyze_protein_hierarchy function to examine whether these bacteria express proteins that are known to induce or accelerate corrosion. This analysis will help determine if these genera possess the molecular machinery to contribute to corrosion despite their absence from corrosion-related literature, potentially identifying novel microbial contributors to corrosion processes that have been overlooked in previous research.

In [ ]:
'''# Creating a dictionary of genera identified as nobel on Notebook 4.
new_genera= ['Oxalobacteraceae_unclassified', 'Oxobacter', 'Mycoplana', 'Bulleidia',  'Oerskovia']
new_genera_df = balanced_markers[balanced_markers["Genus"].isin(new_genera)]
protein_candidates_markers = analyze_protein_hierarchy_rf(new_genera_df, n_estimators=100, n_top_proteins=5, random_state=42)'''
protein_candidates_markers[["protein_name", "genera"]]

In [ ]:
# Creating a dictionary of the protein that were found top on the identified genera to see if these candidate bacteria
# also express that ones.
top_protein= ['3-oxoacyl', 'enoyl', 'beta-ketoacyl', 'glutathione', 'holo', 'aspartate', 'trna (guanine',
              'metal-dependent', 'ferredoxin', 'siroheme-synthase']

# Creating a condition that checks if protein_name starts with any of the strings on top protein
starts_with_condition = new_genera_df["protein_name"].str.lower().apply(
    lambda x: any(x.startswith(tp.lower())for tp in top_protein))
#Apply condition to filter the df
top_protein_in_new = new_genera_df[starts_with_condition]
top_protein_in_new["protein_name"].unique()

In [ ]:
with pd.ExcelWriter(combined_path, mode="a", engine='openpyxl') , if_sheet_exists="replace" as writer:
    protein_markers.to_excel(writer, sheet_name= "protein_markers", index=True, freeze_panes=(1,0))
    all_physicochemical.to_excel(writer, sheet_name= "all_physicochemical", index=True, freeze_panes=(1,0))
    metadata.to_excel(writer, sheet_name= "Metadata", index=True, freeze_panes=(1,0))
    protein_candidates.to_excel(writer, sheet_name= "candidates", index=True, freeze_panes=(1,0))
